# 11 - MMM Model Development

## Objective

Train and compare multiple regression models for Marketing Mix Modeling.

Models covered:
- Multiple Linear Regression
- Ridge Regression
- Lasso Regression
- ElasticNet

The notebook evaluates model performance and interprets coefficients from a business perspective.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

ROOT = Path.cwd()
DATA = ROOT/"data"/"processed"/"marketing_mix_model_ready.csv"

df = pd.read_csv(DATA)

TARGET = "Sales"
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

print(X_train.shape, X_test.shape)


## Train Models

In [ ]:

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1000.0, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=1000.0,l1_ratio=0.5,max_iter=10000)
}

results=[]
predictions={}

for name,model in models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)

    rmse=np.sqrt(mean_squared_error(y_test,pred))

    cv=cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="r2"
    )

    results.append({
        "Model":name,
        "R2":r2_score(y_test,pred),
        "MAE":mean_absolute_error(y_test,pred),
        "RMSE":rmse,
        "CV_R2_Mean":cv.mean()
    })

    predictions[name]=pred

results=pd.DataFrame(results).sort_values("R2",ascending=False)
display(results)


## Best Model

In [ ]:

best_name=results.iloc[0]["Model"]
best=models[best_name]

print("Best Model:",best_name)


## Coefficient Interpretation

In [ ]:

coef=pd.DataFrame({
    "Feature":X.columns,
    "Coefficient":best.coef_
}).sort_values("Coefficient",ascending=False)

display(coef.head(20))


## Feature Importance

In [ ]:

coef_sorted=coef.sort_values("Coefficient")

plt.figure(figsize=(10,8))
plt.barh(coef_sorted["Feature"],coef_sorted["Coefficient"])
plt.title("Model Coefficients")
plt.tight_layout()
plt.show()


## Actual vs Predicted

In [ ]:

pred=predictions[best_name]

plt.figure(figsize=(6,6))
plt.scatter(y_test,pred)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title(best_name)
plt.grid(True)
plt.show()


## Residual Analysis

In [ ]:

residuals=y_test-pred

plt.figure(figsize=(10,4))
plt.hist(residuals,bins=30)
plt.title("Residual Distribution")
plt.show()

plt.figure(figsize=(6,4))
plt.scatter(pred,residuals)
plt.axhline(0,color="red")
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.show()


## Business Interpretation

In [ ]:

positive=coef.query("Coefficient>0").head(10)
negative=coef.query("Coefficient<0").tail(10)

print("Top Positive Drivers")
display(positive)

print("Top Negative Drivers")
display(negative)


## Save Best Model Coefficients

In [ ]:

OUT = ROOT/"models"
OUT.mkdir(exist_ok=True)

coef.to_csv(OUT/"best_model_coefficients.csv",index=False)

print("Saved coefficients.")


# Key Takeaways

- Compare several regression techniques instead of relying on one.
- Regularization (Ridge/Lasso/ElasticNet) often improves stability when marketing variables are correlated.
- Coefficients should always be interpreted together with business knowledge, not in isolation.
